# Vision-Assisted Orbit Correction

In Phase 1, we built a PINN that learns orbital motion purely from Newton's law of gravitation. It works well — but only if the physics we told it about is the *whole* story.

Real satellites experience forces a simple two-body model doesn't capture: atmospheric drag, solar radiation pressure, gravitational quirks from other bodies, or unplanned maneuvers. When that happens, a PINN trained on ideal physics will confidently predict a trajectory that drifts further and further from where the satellite actually is.

This is where **observation** comes in. Real orbit-tracking systems don't rely on physics models alone — they continuously correct their predictions using data from telescopes, radar, or other sensors. In this notebook, we build a simplified version of that idea:

1. Introduce a small, realistic disturbance so the true orbit no longer perfectly matches our PINN's physics.
2. Simulate synthetic "telescope images" of the satellite at its true (disturbed) position.
3. Build a small computer vision model that detects the satellite's position from these images.
4. Use those vision-based observations to correct the PINN's drifting prediction.
5. Compare all three trajectories: the true orbit, the PINN-only prediction, and the vision-corrected prediction.

![Vision-assisted PINN pipeline](../images/pinn_vision_pipeline.png)

## Step 1: Introducing a Disturbance

Our Phase 1 PINN learned pure two-body gravity — no other forces. But real satellites, especially in low orbits, experience **atmospheric drag**: a small force that opposes velocity and slowly bleeds energy from the orbit, causing it to decay over time.

We'll simulate this by adding a simple drag term to our equations of motion:

$$
\vec a_{\text{drag}} = -k \, |\vec v| \, \vec v
$$

In plain terms: drag pushes back against the satellite's motion, and it gets stronger the faster the satellite moves — similar to how wind resistance increases with speed.

We'll generate a **new "true" orbit** using this disturbed physics. Critically, our already-trained PINN knows nothing about drag — it will keep predicting the clean, undisturbed orbit from Phase 1. This mismatch is exactly what we want: a case where pure physics prediction fails, and observation becomes necessary.